# 🛒 E-Commerce Sales Performance & Customer Analytics

**Author:** Roberto Sousa Carranza

Análisis completo de 50,000 transacciones e-commerce combinando **SQL** y **Python**.

## Stack
- **SQL:** PostgreSQL — CTEs, Window Functions (LAG), Cohort Retention
- **Python:** Pandas, NumPy, Scikit-learn
- **Visualización:** Plotly, Matplotlib, Seaborn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

print('✅ Librerías cargadas')

## 1. Carga y Exploración Inicial (EDA)

In [ ]:
df = pd.read_csv('ecommerce_transactions.csv')
df['Transaction_Date'] = pd.to_datetime(df['Transaction_Date'])

print(f'Dimensiones: {df.shape}')
print(f'Período: {df["Transaction_Date"].min()} → {df["Transaction_Date"].max()}')
print(f'Usuarios únicos: {df["User_Name"].nunique()}')
print(f'Países: {df["Country"].nunique()}')
print(f'Categorías: {df["Product_Category"].nunique()}')
print(f'Ingresos totales: ${df["Purchase_Amount"].sum():,.2f}')
print(f'Ticket promedio: ${df["Purchase_Amount"].mean():.2f}')
print(f'Edad promedio clientes: {df["Age"].mean():.1f} años')
print()
print('=== Nulos ===')
print(df.isnull().sum())
print()
print('=== Estadísticas ===')
print(df['Purchase_Amount'].describe())

## 2. Análisis de Ventas por Categoría y País

In [ ]:
# Top categorías por ingresos
cat_ingresos = df.groupby('Product_Category')['Purchase_Amount'].agg(['sum', 'count', 'mean']).sort_values('sum', ascending=False)
cat_ingresos.columns = ['Ingresos', 'Transacciones', 'Ticket_Promedio']

fig = px.bar(cat_ingresos, x=cat_ingresos.index, y='Ingresos',
             title='💰 Ingresos por Categoría de Producto',
             color='Ingresos', color_continuous_scale='Viridis',
             labels={'Product_Category': 'Categoría', 'Ingresos': 'Ingresos ($)'})
fig.update_layout(height=400)
fig.show()

print(cat_ingresos.head(10).to_string())

In [ ]:
# Top países
pais_ingresos = df.groupby('Country')['Purchase_Amount'].sum().sort_values(ascending=False).head(10)

fig = px.bar(pais_ingresos, x=pais_ingresos.values, y=pais_ingresos.index,
             orientation='h', title='🌍 Top 10 Países por Ingresos',
             color=pais_ingresos.values, color_continuous_scale='Blues',
             labels={'x': 'Ingresos ($)', 'y': ''})
fig.update_layout(height=400)
fig.show()

## 3. Análisis Temporal (SQL-style en Python)

In [ ]:
# Replicamos el análisis SQL en Python (CTEs equivalentes)
df['Year'] = df['Transaction_Date'].dt.year
df['Month'] = df['Transaction_Date'].dt.month
df['Quarter'] = df['Transaction_Date'].dt.quarter
df['YearMonth'] = df['Transaction_Date'].dt.to_period('M').astype(str)

# Paso 1: Actividad por usuario/mes (equivalente a la CTE user_activity)
user_activity = df.groupby(['User_Name', 'YearMonth', 'Year', 'Quarter']).size().reset_index(name='transactions')

# Paso 2: LAG para mes anterior (equivalente a retention_calc)
user_activity = user_activity.sort_values(['User_Name', 'YearMonth'])
user_activity['prev_month'] = user_activity.groupby('User_Name')['YearMonth'].shift(1)

# Paso 3: Tasa de retención
def get_prev_ym(ym):
    dt = datetime.strptime(ym, '%Y-%m')
    dt -= timedelta(days=dt.day)
    dt -= timedelta(days=1)
    return dt.strftime('%Y-%m')

user_activity['is_retained'] = user_activity.apply(
    lambda r: r['prev_month'] == get_prev_ym(r['YearMonth']), axis=1
)

retention = user_activity.groupby(['Year', 'Quarter', 'YearMonth']).agg(
    total_customers=('User_Name', 'nunique'),
    retained=('is_retained', 'sum')
).reset_index()
retention['retention_rate'] = (retention['retained'] / retention['total_customers'] * 100).round(1)

fig = px.line(retention, x='YearMonth', y='retention_rate',
              title='📊 Tasa de Retención Mensual (Cohorte)',
              markers=True, line_shape='spline',
              labels={'YearMonth': 'Mes', 'retention_rate': 'Retención (%)'})
fig.update_layout(height=400)
fig.show()

print(retention[['YearMonth', 'total_customers', 'retained', 'retention_rate']].to_string(index=False))

## 4. Métodos de Pago y Estacionalidad

In [ ]:
# Métodos de pago
pay_method = df['Payment_Method'].value_counts().reset_index()
pay_method.columns = ['Método', 'Transacciones']

fig = px.pie(pay_method, values='Transacciones', names='Método',
             title='💳 Distribución de Métodos de Pago',
             color_discrete_sequence=px.colors.qualitative.Set2)
fig.update_layout(height=400)
fig.show()

# Ingresos por trimestre
quarterly = df.groupby(['Year', 'Quarter'])['Purchase_Amount'].sum().reset_index()
quarterly['Label'] = quarterly.apply(lambda r: f'{int(r["Year"])}-Q{int(r["Quarter"])}', axis=1)

fig = px.bar(quarterly, x='Label', y='Purchase_Amount',
             title='📈 Ingresos Trimestrales',
             color='Purchase_Amount', color_continuous_scale='Inferno',
             labels={'Label': 'Trimestre', 'Purchase_Amount': 'Ingresos ($)'})
fig.update_layout(height=400)
fig.show()

print(quarterly[['Label', 'Purchase_Amount']].to_string(index=False))

## 5. Perfil de Clientes por Edad y Segmentación

In [ ]:
# Distribución por edad
fig = px.histogram(df, x='Age', nbins=30, title='👤 Distribución de Edad de Clientes',
                  color_discrete_sequence=['#2E86AB'], opacity=0.7,
                  labels={'Age': 'Edad', 'count': 'Clientes'})
fig.update_layout(height=400)
fig.show()

# Segmentación simple por edad
df['Age_Group'] = pd.cut(df['Age'], bins=[0, 25, 35, 50, 100],
                          labels=['18-25', '26-35', '36-50', '50+'])
age_seg = df.groupby('Age_Group').agg(
    Clientes=('User_Name', 'nunique'),
    Gasto_Promedio=('Purchase_Amount', 'mean'),
    Total_Ingresos=('Purchase_Amount', 'sum')
).round(2)

fig = px.bar(age_seg, x=age_seg.index, y='Total_Ingresos',
             title='💵 Ingresos por Grupo de Edad',
             color='Total_Ingresos', color_continuous_scale='Teal',
             labels={'index': 'Grupo', 'Total_Ingresos': 'Ingresos ($)'})
fig.update_layout(height=400)
fig.show()
print(age_seg)

## 6. Resumen Ejecutivo y Conclusiones

- **Volumen:** 50,000 transacciones analizadas en múltiples países y categorías
- **Retención:** Identificada caída en Q3 (consistente con patrón estacional)
- **Métodos de pago:** Dominan tarjetas (crédito/débito), seguido de PayPal
- **Edad:** Segmento 36-50 años genera mayor ingreso total
- **Categorías:** Electrónica y Ropa lideran en ingresos

Para ver el análisis SQL equivalente, consultar [`retention_analysis.sql`](retention_analysis.sql)